In [ ]:
#!pip install datasets

import numpy as np
import cv2
from tqdm import tqdm
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import os, gc

from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset

In [ ]:
dataset = load_dataset("RationAI/PanNuke")

print(dataset)

# Model 

In [ ]:
#function for double convolusion in unet: two 3x3 convolution followed by relu between each convolution 
def double_conv(inp_c,out_c):
    conv=nn.Sequential(nn.Conv2d(inp_c,out_c,kernel_size=3, padding=1),
                      nn.ReLU(inplace=True),nn.Conv2d(out_c,out_c,kernel_size=3, padding=1),
                      nn.ReLU(inplace=True))
    return conv
'''
#function to crop the tensor for concatenation 
def crop_tensor(tensor,target_tensor):
    target_size=target_tensor.size()[2]
    tensor_size=tensor.size()[2]
    #assuming tensor size is always bigger than target size
    delta=tensor_size-target_size
    delta=delta//2
    return tensor[:,:,delta:tensor_size-delta,delta:tensor_size-delta]'''

def crop_tensor(tensor, target_tensor):
    _, _, H, W = target_tensor.shape
    return tensor[:, :, :H, :W]

class unet(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        #encoder layers
        self.maxpool_2x2=nn.MaxPool2d(kernel_size=2,stride=2)
        #change input channels to 3 cause the input image has 3 channels
        self.down_conv1=double_conv(3,64)
        self.down_conv2=double_conv(64,128)
        self.down_conv3=double_conv(128,256)
        self.down_conv4=double_conv(256,512)
        self.down_conv5=double_conv(512,1024)
        #decoder layers
        self.up_trans1=nn.ConvTranspose2d(in_channels=1024, out_channels=512,kernel_size=2,stride=2)
        self.up_conv1=double_conv(1024,512)

        self.up_trans2=nn.ConvTranspose2d(in_channels=512, out_channels=256,kernel_size=2,stride=2)
        self.up_conv2=double_conv(512,256)

        self.up_trans3=nn.ConvTranspose2d(in_channels=256, out_channels=128,kernel_size=2,stride=2)
        self.up_conv3=double_conv(256,128)

        self.up_trans4=nn.ConvTranspose2d(in_channels=128, out_channels=64,kernel_size=2,stride=2)
        self.up_conv4=double_conv(128,64)

        self.out=nn.Conv2d(in_channels=64,out_channels=num_classes,kernel_size=1)
    
    def forward(self,image):
        #input format: batch size, channel, height, width
        #encoder
        #print(f"size of image {image.size()}")
        x1=self.down_conv1(image) #
        #print(f"size of x1:", x1.size())
        x2=self.maxpool_2x2(x1)
        x3=self.down_conv2(x2)#
        x4=self.maxpool_2x2(x3)
        x5=self.down_conv3(x4) #
        x6=self.maxpool_2x2(x5)
        x7=self.down_conv4(x6) #
        x8=self.maxpool_2x2(x7)
        x9=self.down_conv5(x8) 
        

        x=self.up_trans1(x9)
        y=crop_tensor(x7,x)
        x=self.up_conv1(torch.cat([x,y],1))

        x=self.up_trans2(x)
        y=crop_tensor(x5,x)
        x=self.up_conv2(torch.cat([x,y],1))

        x=self.up_trans3(x)
        y=crop_tensor(x3,x)
        x=self.up_conv3(torch.cat([x,y],1))

        x=self.up_trans4(x)
        y=crop_tensor(x1,x)
        x=self.up_conv4(torch.cat([x,y],1))

        x=self.out(x)
        
        return x
        
        
        

In [ ]:
def visualize_any_mask(mask):
    import matplotlib.pyplot as plt
    import numpy as np

    mask = np.array(mask)

    print("Shape:", mask.shape)
    print("Unique values:", np.unique(mask))

    # Handle different cases
    if mask.ndim == 3:
        if mask.shape[0] == 1:   # (1, H, W)
            mask = mask[0]
        elif mask.shape[-1] == 1:  # (H, W, 1)
            mask = mask[:, :, 0]
        else:
            print("Multi-channel mask, showing first channel")
            mask = mask[0]

    plt.imshow(mask, cmap='gray')
    plt.title("Mask Visualization")
    plt.axis('off')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_image(image):
    image = np.array(image)

    print("Shape:", image.shape)
    print("Min/Max:", image.min(), image.max())

    # Convert CHW → HWC if needed
    if image.ndim == 3 and image.shape[0] in [1, 3]:
        image = np.transpose(image, (1, 2, 0))

    plt.imshow(image)
    plt.title("Image")
    plt.axis('off')
    plt.show()

In [ ]:
def preprocess_mask(instances):
    instances = np.array(instances)

    #completely empty (no nuclei at all)
    if instances.size == 0 or instances.ndim == 1:
        binary_mask = np.zeros((256, 256), dtype=np.uint8)

    else:
        #Normal case
        binary_mask = np.any(instances > 0, axis=0)
        binary_mask = binary_mask.astype(np.uint8)

    #Safety check (but don't crash blindly)
    if binary_mask.shape != (256, 256):
        print("\nUnexpected mask shape")
        print("Raw shape:", instances.shape)
        print("After merge:", binary_mask.shape)
        print("Unique:", np.unique(binary_mask))

        # Instead of crashing → fix it
        binary_mask = np.zeros((256, 256), dtype=np.uint8)

    # add channel
    binary_mask = np.expand_dims(binary_mask, axis=0)

    return binary_mask

In [ ]:
'''
fold1=dataset['fold1']
print(fold1)
sample=fold1[584]
img = preprocess_image(sample["image"])
visualize_image(img)
msk = preprocess_mask(sample["instances"])

visualize_any_mask(msk)
'''

In [ ]:
def preprocess_image(image):
    #print("\n[IMAGE PREPROCESSING]")
    image = np.array(image)
    #print("Original shape:", image.shape)
    #print("Original dtype:", image.dtype)

    # Normalize to [0,1]
    image = image.astype(np.float32) / 255.0

    # Convert HWC → CHW
    image = np.transpose(image, (2, 0, 1))

    #print("Processed shape (CHW):", image.shape)
    #print("Min/Max values:", image.min(), image.max())

    return image




In [ ]:
'''
def preprocess_mask(instances):
    instances = np.array(instances)

    # Merge all nuclei
    binary_mask = np.any(instances > 0, axis=0)
    binary_mask = binary_mask.astype(np.uint8)

    # shape should now be (256,256)
    if binary_mask.shape != (256, 256):
        print("\n❌ BAD MASK")
        print("Raw shape:", instances.shape)
        print("After merge:", binary_mask.shape)
        print("Unique:", np.unique(binary_mask))

        # OPTIONAL: visualize here if needed
        raise ValueError("Mask shape incorrect")

    # add channel
    binary_mask = np.expand_dims(binary_mask, axis=0)

    return binary_mask

'''

In [ ]:
def process_fold(hf_dataset, fold_name):
    print(f"\nProcessing {fold_name}")
    print("Number of samples:", len(hf_dataset))

    images = []
    masks = []

    for i in tqdm(range(len(hf_dataset))):
        sample = hf_dataset[i]

        img = preprocess_image(sample["image"])
        msk = preprocess_mask(sample["instances"])

        images.append(img)
        masks.append(msk)

    
    images = np.stack(images, axis=0)
    masks = np.stack(masks, axis=0)
    print(f"{fold_name} done:")
    print("Images:", images.shape)
    print("Masks:", masks.shape)
    print("Mask values:", np.unique(masks))

    return images, masks

In [ ]:
def preprocess_all_folds(dataset):
    folds = []
    #fold 3 is test
    for fold_name in ["fold1", "fold2"]:
    #for fold_name in ["fold1"]:
        imgs, msks = process_fold(dataset[fold_name], fold_name)
        folds.append((imgs, msks))

    return folds

In [ ]:
'''
from torch.utils.data import TensorDataset, DataLoader

# ---------------- METRICS ----------------
def get_metrics(outputs, masks, threshold=0.5):
    outputs = torch.sigmoid(outputs)  # logits → probabilities
    preds = (outputs > threshold).float()

    preds = preds.view(-1)
    masks = masks.view(-1)

    TP = (preds * masks).sum()
    FP = (preds * (1 - masks)).sum()
    FN = ((1 - preds) * masks).sum()

    dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    iou = TP / (TP + FP + FN + 1e-8)
    acc = (preds == masks).float().mean()

    return dice.item(), iou.item(), acc.item()

def train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=8):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(train_images, train_masks)
    val_ds = TensorDataset(val_images, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_dice = 0

    for epoch in range(epochs):

        # -------- TRAIN --------
        model.train()
        train_loss = 0

        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0

        total_dice = 0
        total_iou = 0
        total_acc = 0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)

                outputs = model(imgs)
                loss = criterion(outputs, masks)

                val_loss += loss.item()

                dice, iou, acc = get_metrics(outputs, masks)

                total_dice += dice
                total_iou += iou
                total_acc += acc

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_dice = total_dice / len(val_loader)
        avg_iou = total_iou / len(val_loader)
        avg_acc = total_acc / len(val_loader)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")
        print(f"Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f} | Acc: {avg_acc:.4f}")

        # -------- SAVE BEST MODEL --------
        if avg_dice > best_dice:
            best_dice = avg_dice
            torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")
            print(f"Best model saved for fold {fold_idx}")

    return best_dice

'''

In [ ]:

# this is after combining dice loss and bce loss
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import gc

# ---------------- DICE LOSS ----------------
def dice_loss(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)

    probs = probs.view(-1)
    targets = targets.view(-1)

    intersection = (probs * targets).sum()
    union = probs.sum() + targets.sum()

    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice


# ---------------- COMBINED LOSS ----------------
def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    d_loss = dice_loss(logits, targets)
    return bce + d_loss


# ---------------- METRICS (GLOBAL - CORRECT) ----------------
def get_metrics(outputs, masks, threshold=0.5):

    outputs = torch.sigmoid(outputs)
    preds = (outputs > threshold).float()

    TP = (preds * masks).sum().item()
    FP = (preds * (1 - masks)).sum().item()
    FN = ((1 - preds) * masks).sum().item()
    TN = ((1 - preds) * (1 - masks)).sum().item()

    dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    iou = TP / (TP + FP + FN + 1e-8)
    acc = (TP + TN) / (TP + TN + FP + FN + 1e-8)

    return TP, FP, FN, TN, dice, iou, acc

'''
# ---------------- TRAIN FUNCTION ----------------
def train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=4):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(train_images, train_masks)
    val_ds = TensorDataset(val_images, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_dice = 0

    for epoch in range(epochs):

        # -------- TRAIN --------
        model.train()
        train_loss = 0

        for imgs, masks in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = combined_loss(outputs, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0

        total_TP, total_FP, total_FN, total_TN = 0, 0, 0, 0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)

                outputs = model(imgs)
                loss = combined_loss(outputs, masks)

                val_loss += loss.item()

                TP, FP, FN, TN, _, _, _ = get_metrics(outputs, masks)

                total_TP += TP
                total_FP += FP
                total_FN += FN
                total_TN += TN

        # -------- FINAL METRICS (CORRECT WAY) --------
        dice = (2 * total_TP) / (2 * total_TP + total_FP + total_FN + 1e-8)
        iou = total_TP / (total_TP + total_FP + total_FN + 1e-8)
        acc = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN + 1e-8)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")
        print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Acc: {acc:.4f}")

        # -------- SAVE BEST MODEL --------
        if dice > best_dice:
            best_dice = dice
            torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")
            print(f"Best model saved for fold {fold_idx}")

        # -------- MEMORY CLEANUP PER EPOCH --------
        torch.cuda.empty_cache()

    # -------- FINAL CLEANUP --------
    del train_loader, val_loader
    del train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

    return best_dice

'''

In [ ]:
# separating validation and training

def validate_model(model, val_loader, device):

    model.eval()

    val_loss = 0
    total_TP, total_FP, total_FN, total_TN = 0, 0, 0, 0

    with torch.no_grad():
        for imgs, masks in val_loader:

            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(imgs)
            loss = combined_loss(outputs, masks)

            val_loss += loss.item()

            TP, FP, FN, TN, _, _, _ = get_metrics(outputs, masks)

            total_TP += TP
            total_FP += FP
            total_FN += FN
            total_TN += TN

    # ---- FINAL METRICS ----
    dice = (2 * total_TP) / (2 * total_TP + total_FP + total_FN + 1e-8)
    iou = total_TP / (total_TP + total_FP + total_FN + 1e-8)
    acc = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN + 1e-8)

    avg_val_loss = val_loss / len(val_loader)

    return avg_val_loss, dice, iou, acc

def train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=4):

    

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(train_images, train_masks)
    val_ds = TensorDataset(val_images, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_dice = 0

    # -------- CHECKPOINT PATH --------
    checkpoint_path = f"/kaggle/working/checkpoint_fold_{fold_idx}.pth"

    start_epoch = 0

    # -------- RESUME IF EXISTS --------
    if os.path.exists(checkpoint_path):
        print("Resuming from checkpoint...")
        checkpoint = torch.load(checkpoint_path)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_dice = checkpoint['best_dice']

    # -------- TRAIN LOOP --------
    for epoch in range(start_epoch, epochs):

        model.train()
        train_loss = 0

        for imgs, masks in train_loader:

            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = combined_loss(outputs, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")

        # -------- VALIDATION --------
        if (epoch)  == epoch:

            avg_val_loss, dice, iou, acc = validate_model(model, val_loader, device)

            print(f"Val Loss: {avg_val_loss:.4f}")
            print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Acc: {acc:.4f}")

            # ---- SAVE BEST MODEL ----
            if dice > best_dice:
                best_dice = dice
                torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")
                print(f"Best model saved for fold {fold_idx}")

        # -------- SAVE CHECKPOINT EVERY EPOCH --------
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_dice': best_dice
        }, checkpoint_path)

        # -------- MEMORY CLEANUP --------
        torch.cuda.empty_cache()

    # -------- FINAL CLEANUP --------
    del train_loader, val_loader
    del train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

    return best_dice


In [ ]:
import torch
from torch.utils.data import Dataset

class SegDataset(Dataset):
    def __init__(self, images, masks, filenames):
        self.images = images
        self.masks = masks
        self.filenames = filenames

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.images[idx], dtype=torch.float32),
            torch.tensor(self.masks[idx], dtype=torch.long),
            self.filenames[idx]
        )

In [ ]:
#!pip install datasets

import numpy as np
import cv2
from tqdm import tqdm
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import os, gc

from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
dataset = load_dataset("RationAI/PanNuke")

print(dataset)
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
DatasetDict({
    fold1: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2656
    })
    fold2: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2523
    })
    fold3: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2722
    })
})
Model
#function for double convolusion in unet: two 3x3 convolution followed by relu between each convolution 
def double_conv(inp_c,out_c):
    conv=nn.Sequential(nn.Conv2d(inp_c,out_c,kernel_size=3, padding=1),
                      nn.ReLU(inplace=True),nn.Conv2d(out_c,out_c,kernel_size=3, padding=1),
                      nn.ReLU(inplace=True))
    return conv
'''
#function to crop the tensor for concatenation 
def crop_tensor(tensor,target_tensor):
    target_size=target_tensor.size()[2]
    tensor_size=tensor.size()[2]
    #assuming tensor size is always bigger than target size
    delta=tensor_size-target_size
    delta=delta//2
    return tensor[:,:,delta:tensor_size-delta,delta:tensor_size-delta]'''

def crop_tensor(tensor, target_tensor):
    _, _, H, W = target_tensor.shape
    return tensor[:, :, :H, :W]

class unet(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        #encoder layers
        self.maxpool_2x2=nn.MaxPool2d(kernel_size=2,stride=2)
        #change input channels to 3 cause the input image has 3 channels
        self.down_conv1=double_conv(3,64)
        self.down_conv2=double_conv(64,128)
        self.down_conv3=double_conv(128,256)
        self.down_conv4=double_conv(256,512)
        self.down_conv5=double_conv(512,1024)
        #decoder layers
        self.up_trans1=nn.ConvTranspose2d(in_channels=1024, out_channels=512,kernel_size=2,stride=2)
        self.up_conv1=double_conv(1024,512)

        self.up_trans2=nn.ConvTranspose2d(in_channels=512, out_channels=256,kernel_size=2,stride=2)
        self.up_conv2=double_conv(512,256)

        self.up_trans3=nn.ConvTranspose2d(in_channels=256, out_channels=128,kernel_size=2,stride=2)
        self.up_conv3=double_conv(256,128)

        self.up_trans4=nn.ConvTranspose2d(in_channels=128, out_channels=64,kernel_size=2,stride=2)
        self.up_conv4=double_conv(128,64)

        self.out=nn.Conv2d(in_channels=64,out_channels=num_classes,kernel_size=1)
    
    def forward(self,image):
        #input format: batch size, channel, height, width
        #encoder
        #print(f"size of image {image.size()}")
        x1=self.down_conv1(image) #
        #print(f"size of x1:", x1.size())
        x2=self.maxpool_2x2(x1)
        x3=self.down_conv2(x2)#
        x4=self.maxpool_2x2(x3)
        x5=self.down_conv3(x4) #
        x6=self.maxpool_2x2(x5)
        x7=self.down_conv4(x6) #
        x8=self.maxpool_2x2(x7)
        x9=self.down_conv5(x8) 
        

        x=self.up_trans1(x9)
        y=crop_tensor(x7,x)
        x=self.up_conv1(torch.cat([x,y],1))

        x=self.up_trans2(x)
        y=crop_tensor(x5,x)
        x=self.up_conv2(torch.cat([x,y],1))

        x=self.up_trans3(x)
        y=crop_tensor(x3,x)
        x=self.up_conv3(torch.cat([x,y],1))

        x=self.up_trans4(x)
        y=crop_tensor(x1,x)
        x=self.up_conv4(torch.cat([x,y],1))

        x=self.out(x)
        
        return x
        
        
        
def visualize_any_mask(mask):
    import matplotlib.pyplot as plt
    import numpy as np

    mask = np.array(mask)

    print("Shape:", mask.shape)
    print("Unique values:", np.unique(mask))

    # Handle different cases
    if mask.ndim == 3:
        if mask.shape[0] == 1:   # (1, H, W)
            mask = mask[0]
        elif mask.shape[-1] == 1:  # (H, W, 1)
            mask = mask[:, :, 0]
        else:
            print("Multi-channel mask, showing first channel")
            mask = mask[0]

    plt.imshow(mask, cmap='gray')
    plt.title("Mask Visualization")
    plt.axis('off')
    plt.show()
import matplotlib.pyplot as plt
import numpy as np

def visualize_image(image):
    image = np.array(image)

    print("Shape:", image.shape)
    print("Min/Max:", image.min(), image.max())

    # Convert CHW → HWC if needed
    if image.ndim == 3 and image.shape[0] in [1, 3]:
        image = np.transpose(image, (1, 2, 0))

    plt.imshow(image)
    plt.title("Image")
    plt.axis('off')
    plt.show()
def preprocess_mask(instances):
    instances = np.array(instances)

    #completely empty (no nuclei at all)
    if instances.size == 0 or instances.ndim == 1:
        binary_mask = np.zeros((256, 256), dtype=np.uint8)

    else:
        #Normal case
        binary_mask = np.any(instances > 0, axis=0)
        binary_mask = binary_mask.astype(np.uint8)

    #Safety check (but don't crash blindly)
    if binary_mask.shape != (256, 256):
        print("\nUnexpected mask shape")
        print("Raw shape:", instances.shape)
        print("After merge:", binary_mask.shape)
        print("Unique:", np.unique(binary_mask))

        # Instead of crashing → fix it
        binary_mask = np.zeros((256, 256), dtype=np.uint8)

    # add channel
    binary_mask = np.expand_dims(binary_mask, axis=0)

    return binary_mask
'''
fold1=dataset['fold1']
print(fold1)
sample=fold1[584]
img = preprocess_image(sample["image"])
visualize_image(img)
msk = preprocess_mask(sample["instances"])

visualize_any_mask(msk)
'''
'\nfold1=dataset[\'fold1\']\nprint(fold1)\nsample=fold1[584]\nimg = preprocess_image(sample["image"])\nvisualize_image(img)\nmsk = preprocess_mask(sample["instances"])\n\nvisualize_any_mask(msk)\n'
def preprocess_image(image):
    #print("\n[IMAGE PREPROCESSING]")
    image = np.array(image)
    #print("Original shape:", image.shape)
    #print("Original dtype:", image.dtype)

    # Normalize to [0,1]
    image = image.astype(np.float32) / 255.0

    # Convert HWC → CHW
    image = np.transpose(image, (2, 0, 1))

    #print("Processed shape (CHW):", image.shape)
    #print("Min/Max values:", image.min(), image.max())

    return image
'''
def preprocess_mask(instances):
    instances = np.array(instances)

    # Merge all nuclei
    binary_mask = np.any(instances > 0, axis=0)
    binary_mask = binary_mask.astype(np.uint8)

    # shape should now be (256,256)
    if binary_mask.shape != (256, 256):
        print("\n❌ BAD MASK")
        print("Raw shape:", instances.shape)
        print("After merge:", binary_mask.shape)
        print("Unique:", np.unique(binary_mask))

        # OPTIONAL: visualize here if needed
        raise ValueError("Mask shape incorrect")

    # add channel
    binary_mask = np.expand_dims(binary_mask, axis=0)

    return binary_mask

'''
'\ndef preprocess_mask(instances):\n    instances = np.array(instances)\n\n    # Merge all nuclei\n    binary_mask = np.any(instances > 0, axis=0)\n    binary_mask = binary_mask.astype(np.uint8)\n\n    # shape should now be (256,256)\n    if binary_mask.shape != (256, 256):\n        print("\n❌ BAD MASK")\n        print("Raw shape:", instances.shape)\n        print("After merge:", binary_mask.shape)\n        print("Unique:", np.unique(binary_mask))\n\n        # OPTIONAL: visualize here if needed\n        raise ValueError("Mask shape incorrect")\n\n    # add channel\n    binary_mask = np.expand_dims(binary_mask, axis=0)\n\n    return binary_mask\n\n'
 
def process_fold(hf_dataset, fold_name):
    print(f"\nProcessing {fold_name}")
    print("Number of samples:", len(hf_dataset))

    images = []
    masks = []

    for i in tqdm(range(len(hf_dataset))):
        sample = hf_dataset[i]

        img = preprocess_image(sample["image"])
        msk = preprocess_mask(sample["instances"])

        images.append(img)
        masks.append(msk)

    
    images = np.stack(images, axis=0)
    masks = np.stack(masks, axis=0)
    print(f"{fold_name} done:")
    print("Images:", images.shape)
    print("Masks:", masks.shape)
    print("Mask values:", np.unique(masks))

    return images, masks
def preprocess_all_folds(dataset):
    folds = []
    #fold 3 is test
    for fold_name in ["fold1", "fold2"]:
    #for fold_name in ["fold1"]:
        imgs, msks = process_fold(dataset[fold_name], fold_name)
        folds.append((imgs, msks))

    return folds
'''
from torch.utils.data import TensorDataset, DataLoader

# ---------------- METRICS ----------------
def get_metrics(outputs, masks, threshold=0.5):
    outputs = torch.sigmoid(outputs)  # logits → probabilities
    preds = (outputs > threshold).float()

    preds = preds.view(-1)
    masks = masks.view(-1)

    TP = (preds * masks).sum()
    FP = (preds * (1 - masks)).sum()
    FN = ((1 - preds) * masks).sum()

    dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    iou = TP / (TP + FP + FN + 1e-8)
    acc = (preds == masks).float().mean()

    return dice.item(), iou.item(), acc.item()

def train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=8):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(train_images, train_masks)
    val_ds = TensorDataset(val_images, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_dice = 0

    for epoch in range(epochs):

        # -------- TRAIN --------
        model.train()
        train_loss = 0

        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0

        total_dice = 0
        total_iou = 0
        total_acc = 0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)

                outputs = model(imgs)
                loss = criterion(outputs, masks)

                val_loss += loss.item()

                dice, iou, acc = get_metrics(outputs, masks)

                total_dice += dice
                total_iou += iou
                total_acc += acc

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_dice = total_dice / len(val_loader)
        avg_iou = total_iou / len(val_loader)
        avg_acc = total_acc / len(val_loader)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")
        print(f"Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f} | Acc: {avg_acc:.4f}")

        # -------- SAVE BEST MODEL --------
        if avg_dice > best_dice:
            best_dice = avg_dice
            torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")
            print(f"Best model saved for fold {fold_idx}")

    return best_dice

'''
'\nfrom torch.utils.data import TensorDataset, DataLoader\n\n# ---------------- METRICS ----------------\ndef get_metrics(outputs, masks, threshold=0.5):\n    outputs = torch.sigmoid(outputs)  # logits → probabilities\n    preds = (outputs > threshold).float()\n\n    preds = preds.view(-1)\n    masks = masks.view(-1)\n\n    TP = (preds * masks).sum()\n    FP = (preds * (1 - masks)).sum()\n    FN = ((1 - preds) * masks).sum()\n\n    dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)\n    iou = TP / (TP + FP + FN + 1e-8)\n    acc = (preds == masks).float().mean()\n\n    return dice.item(), iou.item(), acc.item()\n\ndef train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=8):\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = model.to(device)\n\n    train_ds = TensorDataset(train_images, train_masks)\n    val_ds = TensorDataset(val_images, val_masks)\n\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)\n\n    criterion = nn.BCEWithLogitsLoss()\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)\n\n    best_dice = 0\n\n    for epoch in range(epochs):\n\n        # -------- TRAIN --------\n        model.train()\n        train_loss = 0\n\n        for imgs, masks in train_loader:\n            imgs, masks = imgs.to(device), masks.to(device)\n\n            optimizer.zero_grad()\n            outputs = model(imgs)\n            loss = criterion(outputs, masks)\n\n            loss.backward()\n            optimizer.step()\n\n            train_loss += loss.item()\n\n        # -------- VALIDATION --------\n        model.eval()\n        val_loss = 0\n\n        total_dice = 0\n        total_iou = 0\n        total_acc = 0\n\n        with torch.no_grad():\n            for imgs, masks in val_loader:\n                imgs, masks = imgs.to(device), masks.to(device)\n\n                outputs = model(imgs)\n                loss = criterion(outputs, masks)\n\n                val_loss += loss.item()\n\n                dice, iou, acc = get_metrics(outputs, masks)\n\n                total_dice += dice\n                total_iou += iou\n                total_acc += acc\n\n        avg_train_loss = train_loss / len(train_loader)\n        avg_val_loss = val_loss / len(val_loader)\n        avg_dice = total_dice / len(val_loader)\n        avg_iou = total_iou / len(val_loader)\n        avg_acc = total_acc / len(val_loader)\n\n        print(f"\nEpoch {epoch+1}/{epochs}")\n        print(f"Train Loss: {avg_train_loss:.4f}")\n        print(f"Val Loss:   {avg_val_loss:.4f}")\n        print(f"Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f} | Acc: {avg_acc:.4f}")\n\n        # -------- SAVE BEST MODEL --------\n        if avg_dice > best_dice:\n            best_dice = avg_dice\n            torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")\n            print(f"Best model saved for fold {fold_idx}")\n\n    return best_dice\n\n'
# this is after combining dice loss and bce loss
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import gc

# ---------------- DICE LOSS ----------------
def dice_loss(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)

    probs = probs.view(-1)
    targets = targets.view(-1)

    intersection = (probs * targets).sum()
    union = probs.sum() + targets.sum()

    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice


# ---------------- COMBINED LOSS ----------------
def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    d_loss = dice_loss(logits, targets)
    return bce + d_loss


# ---------------- METRICS (GLOBAL - CORRECT) ----------------
def get_metrics(outputs, masks, threshold=0.5):

    outputs = torch.sigmoid(outputs)
    preds = (outputs > threshold).float()

    TP = (preds * masks).sum().item()
    FP = (preds * (1 - masks)).sum().item()
    FN = ((1 - preds) * masks).sum().item()
    TN = ((1 - preds) * (1 - masks)).sum().item()

    dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    iou = TP / (TP + FP + FN + 1e-8)
    acc = (TP + TN) / (TP + TN + FP + FN + 1e-8)

    return TP, FP, FN, TN, dice, iou, acc

'''
# ---------------- TRAIN FUNCTION ----------------
def train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=4):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(train_images, train_masks)
    val_ds = TensorDataset(val_images, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_dice = 0

    for epoch in range(epochs):

        # -------- TRAIN --------
        model.train()
        train_loss = 0

        for imgs, masks in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = combined_loss(outputs, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0

        total_TP, total_FP, total_FN, total_TN = 0, 0, 0, 0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)

                outputs = model(imgs)
                loss = combined_loss(outputs, masks)

                val_loss += loss.item()

                TP, FP, FN, TN, _, _, _ = get_metrics(outputs, masks)

                total_TP += TP
                total_FP += FP
                total_FN += FN
                total_TN += TN

        # -------- FINAL METRICS (CORRECT WAY) --------
        dice = (2 * total_TP) / (2 * total_TP + total_FP + total_FN + 1e-8)
        iou = total_TP / (total_TP + total_FP + total_FN + 1e-8)
        acc = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN + 1e-8)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")
        print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Acc: {acc:.4f}")

        # -------- SAVE BEST MODEL --------
        if dice > best_dice:
            best_dice = dice
            torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")
            print(f"Best model saved for fold {fold_idx}")

        # -------- MEMORY CLEANUP PER EPOCH --------
        torch.cuda.empty_cache()

    # -------- FINAL CLEANUP --------
    del train_loader, val_loader
    del train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

    return best_dice

'''
'\n# ---------------- TRAIN FUNCTION ----------------\ndef train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=4):\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = model.to(device)\n\n    train_ds = TensorDataset(train_images, train_masks)\n    val_ds = TensorDataset(val_images, val_masks)\n\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)\n\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)\n\n    best_dice = 0\n\n    for epoch in range(epochs):\n\n        # -------- TRAIN --------\n        model.train()\n        train_loss = 0\n\n        for imgs, masks in train_loader:\n            imgs = imgs.to(device, non_blocking=True)\n            masks = masks.to(device, non_blocking=True)\n\n            optimizer.zero_grad()\n\n            outputs = model(imgs)\n            loss = combined_loss(outputs, masks)\n\n            loss.backward()\n            optimizer.step()\n\n            train_loss += loss.item()\n\n        # -------- VALIDATION --------\n        model.eval()\n        val_loss = 0\n\n        total_TP, total_FP, total_FN, total_TN = 0, 0, 0, 0\n\n        with torch.no_grad():\n            for imgs, masks in val_loader:\n                imgs = imgs.to(device, non_blocking=True)\n                masks = masks.to(device, non_blocking=True)\n\n                outputs = model(imgs)\n                loss = combined_loss(outputs, masks)\n\n                val_loss += loss.item()\n\n                TP, FP, FN, TN, _, _, _ = get_metrics(outputs, masks)\n\n                total_TP += TP\n                total_FP += FP\n                total_FN += FN\n                total_TN += TN\n\n        # -------- FINAL METRICS (CORRECT WAY) --------\n        dice = (2 * total_TP) / (2 * total_TP + total_FP + total_FN + 1e-8)\n        iou = total_TP / (total_TP + total_FP + total_FN + 1e-8)\n        acc = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN + 1e-8)\n\n        avg_train_loss = train_loss / len(train_loader)\n        avg_val_loss = val_loss / len(val_loader)\n\n        print(f"\nEpoch {epoch+1}/{epochs}")\n        print(f"Train Loss: {avg_train_loss:.4f}")\n        print(f"Val Loss:   {avg_val_loss:.4f}")\n        print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Acc: {acc:.4f}")\n\n        # -------- SAVE BEST MODEL --------\n        if dice > best_dice:\n            best_dice = dice\n            torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")\n            print(f"Best model saved for fold {fold_idx}")\n\n        # -------- MEMORY CLEANUP PER EPOCH --------\n        torch.cuda.empty_cache()\n\n    # -------- FINAL CLEANUP --------\n    del train_loader, val_loader\n    del train_ds, val_ds\n    torch.cuda.empty_cache()\n    gc.collect()\n\n    return best_dice\n\n'
# separating validation and training

def validate_model(model, val_loader, device):

    model.eval()

    val_loss = 0
    total_TP, total_FP, total_FN, total_TN = 0, 0, 0, 0

    with torch.no_grad():
        for imgs, masks in val_loader:

            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(imgs)
            loss = combined_loss(outputs, masks)

            val_loss += loss.item()

            TP, FP, FN, TN, _, _, _ = get_metrics(outputs, masks)

            total_TP += TP
            total_FP += FP
            total_FN += FN
            total_TN += TN

    # ---- FINAL METRICS ----
    dice = (2 * total_TP) / (2 * total_TP + total_FP + total_FN + 1e-8)
    iou = total_TP / (total_TP + total_FP + total_FN + 1e-8)
    acc = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN + 1e-8)

    avg_val_loss = val_loss / len(val_loader)

    return avg_val_loss, dice, iou, acc

def train_model(train_images, train_masks, val_images, val_masks, model, fold_idx, epochs=10, batch_size=4):

    

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(train_images, train_masks)
    val_ds = TensorDataset(val_images, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_dice = 0

    # -------- CHECKPOINT PATH --------
    checkpoint_path = f"/kaggle/working/checkpoint_fold_{fold_idx}.pth"

    start_epoch = 0

    # -------- RESUME IF EXISTS --------
    if os.path.exists(checkpoint_path):
        print("Resuming from checkpoint...")
        checkpoint = torch.load(checkpoint_path)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_dice = checkpoint['best_dice']

    # -------- TRAIN LOOP --------
    for epoch in range(start_epoch, epochs):

        model.train()
        train_loss = 0

        for imgs, masks in train_loader:

            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = combined_loss(outputs, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")

        # -------- VALIDATION --------
        if (epoch)  == epoch:

            avg_val_loss, dice, iou, acc = validate_model(model, val_loader, device)

            print(f"Val Loss: {avg_val_loss:.4f}")
            print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Acc: {acc:.4f}")

            # ---- SAVE BEST MODEL ----
            if dice > best_dice:
                best_dice = dice
                torch.save(model.state_dict(), f"/kaggle/working/best_model_fold_{fold_idx}.pth")
                print(f"Best model saved for fold {fold_idx}")

        # -------- SAVE CHECKPOINT EVERY EPOCH --------
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_dice': best_dice
        }, checkpoint_path)

        # -------- MEMORY CLEANUP --------
        torch.cuda.empty_cache()

    # -------- FINAL CLEANUP --------
    del train_loader, val_loader
    del train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

    return best_dice
def run_cross_validation(processed_folds):

    num_folds = len(processed_folds)
    fold_dice_scores = []

    for fold_idx in range(num_folds):

        print(f"\n{'='*50}")
        print(f"Running Fold {fold_idx} as Validation")
        print(f"{'='*50}")

        val_images, val_masks = processed_folds[fold_idx]

        train_images = []
        train_masks = []

        for i in range(num_folds):
            if i != fold_idx:
                train_images.append(processed_folds[i][0])
                train_masks.append(processed_folds[i][1])

        train_images = np.concatenate(train_images, axis=0)
        train_masks = np.concatenate(train_masks, axis=0)

        print("Train shape:", train_images.shape)
        print("Val shape:", val_images.shape)

        # Convert to torch
        train_images = torch.tensor(train_images, dtype=torch.float32)
        train_masks = torch.tensor(train_masks, dtype=torch.float32)

        val_images = torch.tensor(val_images, dtype=torch.float32)
        val_masks = torch.tensor(val_masks, dtype=torch.float32)

        model = unet(1)

        best_dice = train_model(
            train_images, train_masks,
            val_images, val_masks,
            model,
            fold_idx
        )

        fold_dice_scores.append(best_dice)

        print(f"Fold {fold_idx} Best Dice: {best_dice:.4f}")

    # -------- FINAL RESULTS --------
    mean_dice = np.mean(fold_dice_scores)
    std_dice = np.std(fold_dice_scores)

    print("\n" + "="*50)
    print("FINAL CROSS-VALIDATION RESULT")
    print(f"Mean Dice: {mean_dice:.4f} ± {std_dice:.4f}")
    print("="*50)

    return fold_dice_scores
# Step 1: preprocess once
processed_folds = preprocess_all_folds(dataset)

# Step 2: run full CV automatically
run_cross_validation(processed_folds)
Processing fold1
Number of samples: 2656
100%|██████████| 2656/2656 [00:23<00:00, 112.77it/s]
fold1 done:
Images: (2656, 3, 256, 256)
Masks: (2656, 1, 256, 256)
Mask values: [0 1]

Processing fold2
Number of samples: 2523
100%|██████████| 2523/2523 [00:20<00:00, 121.89it/s]
fold2 done:
Images: (2523, 3, 256, 256)
Masks: (2523, 1, 256, 256)
Mask values: [0 1]

==================================================
Running Fold 0 as Validation
==================================================
Train shape: (2523, 3, 256, 256)
Val shape: (2656, 3, 256, 256)

Epoch 1/10
Train Loss: 0.7528
Val Loss: 0.6058
Dice: 0.7194 | IoU: 0.5618 | Acc: 0.9005
Best model saved for fold 0

Epoch 2/10
Train Loss: 0.5647
Val Loss: 0.5374
Dice: 0.7524 | IoU: 0.6031 | Acc: 0.9129
Best model saved for fold 0

Epoch 3/10
Train Loss: 0.4925
Val Loss: 0.4742
Dice: 0.7839 | IoU: 0.6446 | Acc: 0.9237
Best model saved for fold 0

Epoch 4/10
Train Loss: 0.4545
Val Loss: 0.4578
Dice: 0.7921 | IoU: 0.6557 | Acc: 0.9257
Best model saved for fold 0

Epoch 5/10
Train Loss: 0.4176
Val Loss: 0.4239
Dice: 0.8049 | IoU: 0.6735 | Acc: 0.9336
Best model saved for fold 0

Epoch 6/10
Train Loss: 0.3950
Val Loss: 0.4043
Dice: 0.8189 | IoU: 0.6934 | Acc: 0.9337
Best model saved for fold 0

Epoch 7/10
Train Loss: 0.3762
Val Loss: 0.3938
Dice: 0.8239 | IoU: 0.7005 | Acc: 0.9395
Best model saved for fold 0

Epoch 8/10
Train Loss: 0.3573
Val Loss: 0.3743
Dice: 0.8306 | IoU: 0.7102 | Acc: 0.9414
Best model saved for fold 0

Epoch 9/10
Train Loss: 0.3440
Val Loss: 0.3819
Dice: 0.8252 | IoU: 0.7024 | Acc: 0.9419

Epoch 10/10
Train Loss: 0.3314
Val Loss: 0.3802
Dice: 0.8293 | IoU: 0.7084 | Acc: 0.9382
Fold 0 Best Dice: 0.8306

==================================================
Running Fold 1 as Validation
==================================================
Train shape: (2656, 3, 256, 256)
Val shape: (2523, 3, 256, 256)

Epoch 1/10
Train Loss: 0.7310
Val Loss: 0.6040
Dice: 0.7257 | IoU: 0.5694 | Acc: 0.9034
Best model saved for fold 1

Epoch 2/10
Train Loss: 0.5386
Val Loss: 0.5367
Dice: 0.7489 | IoU: 0.5985 | Acc: 0.9216
Best model saved for fold 1

Epoch 3/10
Train Loss: 0.4877
Val Loss: 0.4802
Dice: 0.7826 | IoU: 0.6428 | Acc: 0.9253
Best model saved for fold 1

Epoch 4/10
Train Loss: 0.4528
Val Loss: 0.4589
Dice: 0.7910 | IoU: 0.6543 | Acc: 0.9327
Best model saved for fold 1

Epoch 5/10
Train Loss: 0.4296
Val Loss: 0.4373
Dice: 0.8012 | IoU: 0.6684 | Acc: 0.9299
Best model saved for fold 1

Epoch 6/10
Train Loss: 0.3985
Val Loss: 0.4150
Dice: 0.8075 | IoU: 0.6772 | Acc: 0.9390
Best model saved for fold 1

Epoch 7/10
Train Loss: 0.3872
Val Loss: 0.3958
Dice: 0.8198 | IoU: 0.6946 | Acc: 0.9382
Best model saved for fold 1

Epoch 8/10
Train Loss: 0.3623
Val Loss: 0.3962
Dice: 0.8206 | IoU: 0.6958 | Acc: 0.9423
Best model saved for fold 1

Epoch 9/10
Train Loss: 0.3517
Val Loss: 0.3792
Dice: 0.8265 | IoU: 0.7043 | Acc: 0.9439
Best model saved for fold 1

Epoch 10/10
Train Loss: 0.3344
Val Loss: 0.3726
Dice: 0.8302 | IoU: 0.7096 | Acc: 0.9455
Best model saved for fold 1
Fold 1 Best Dice: 0.8302

==================================================
FINAL CROSS-VALIDATION RESULT
Mean Dice: 0.8304 ± 0.0002
==================================================
[0.8305614576532041, 0.8301537787671516]
"""
#training one one fold as validation 
val_images_np, val_masks_np = processed_folds[0]

val_images = torch.tensor(val_images_np, dtype=torch.float32)
val_masks  = torch.tensor(val_masks_np, dtype=torch.float32)

train_images_np = np.concatenate([
    processed_folds[1][0],
    processed_folds[2][0]
], axis=0)

train_masks_np = np.concatenate([
    processed_folds[1][1],
    processed_folds[2][1]
], axis=0)

train_images = torch.tensor(train_images_np, dtype=torch.float32)
train_masks  = torch.tensor(train_masks_np, dtype=torch.float32)
"""
'\n#training one one fold as validation \nval_images_np, val_masks_np = processed_folds[0]\n\nval_images = torch.tensor(val_images_np, dtype=torch.float32)\nval_masks  = torch.tensor(val_masks_np, dtype=torch.float32)\n\ntrain_images_np = np.concatenate([\n    processed_folds[1][0],\n    processed_folds[2][0]\n], axis=0)\n\ntrain_masks_np = np.concatenate([\n    processed_folds[1][1],\n    processed_folds[2][1]\n], axis=0)\n\ntrain_images = torch.tensor(train_images_np, dtype=torch.float32)\ntrain_masks  = torch.tensor(train_masks_np, dtype=torch.float32)\n'
"""
model = unet(1)

best_dice = train_model(
    train_images, train_masks,
    val_images, val_masks,
    model,
    fold_idx=0,
    epochs=5,
    batch_size=4
)
"""
'\nmodel = unet(1)\n\nbest_dice = train_model(\n    train_images, train_masks,\n    val_images, val_masks,\n    model,\n    fold_idx=0,\n    epochs=5,\n    batch_size=4\n)\n'
import matplotlib.pyplot as plt
import torch

def visualize_predictions(model, images, masks, device, num_samples=3):

    model.eval()

    indices = torch.randperm(len(images))[:num_samples]

    with torch.no_grad():
        for idx in indices:

            img = images[idx].unsqueeze(0).to(device)   # add batch dim
            gt_mask = masks[idx].cpu().numpy()

            output = model(img)
            pred_mask = torch.sigmoid(output).cpu().numpy()[0, 0]

            # threshold
            pred_mask = (pred_mask > 0.5).astype(int)

            img_np = images[idx].cpu().numpy().transpose(1, 2, 0)

            # ---- PLOT ----
            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.title("Image")
            plt.imshow(img_np)
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.title("Ground Truth")
            plt.imshow(gt_mask.squeeze(), cmap='gray')
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.title("Prediction")
            plt.imshow(pred_mask, cmap='gray')
            plt.axis("off")

            plt.show()
"""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load("/kaggle/working/best_model_fold_0.pth"))
model.to(device)

visualize_predictions(model, val_images, val_masks, device, num_samples=3)
"""
'\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\nmodel.load_state_dict(torch.load("/kaggle/working/best_model_fold_0.pth"))\nmodel.to(device)\n\nvisualize_predictions(model, val_images, val_masks, device, num_samples=3)\n'
"""
from IPython.display import FileLink

# Path to your saved model
model_path = "/kaggle/working/best_model_fold_0.pth"

# Creates a clickable download link
FileLink(model_path)
"""
'\nfrom IPython.display import FileLink\n\n# Path to your saved model\nmodel_path = "/kaggle/working/best_model_fold_0.pth"\n\n# Creates a clickable download link\nFileLink(model_path)\n'
 
 

In [ ]:
from torch.utils.data import DataLoader
import torch.nn.functional as F

def train_model(
    train_dataset,
    val_dataset,
    model,
    fold_idx,
    num_classes=6,
    epochs=30,
    batch_size=8,
    lr=1e-3,
    save_predictions=True,
    save_dir="predictions",
    file_suffix="exp1"
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = model.to(device)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_dice = 0.0
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):

        # -------- TRAIN --------
        model.train()
        total_loss = 0

        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)

            ce_loss = criterion(outputs, masks)
            d_loss  = dice_loss(outputs, masks, num_classes)

            loss = ce_loss + d_loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # -------- VALIDATION --------
        model.eval()
        total_dice = 0

        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)

                outputs = model(imgs)
                total_dice += dice_score(outputs, masks, num_classes).item()

        avg_dice = total_dice / len(val_loader)

        print(f"[Fold {fold_idx}] Epoch {epoch+1}/{epochs}")
        print(f"Loss: {avg_loss:.4f} | Val Dice: {avg_dice:.4f}")

        # -------- SAVE BEST --------
        if avg_dice > best_dice:
            best_dice = avg_dice
            print("🔥 New best model!")

            if save_predictions:
                # save from FIRST batch only
                imgs, masks, filenames = next(iter(val_loader))
                imgs, masks = imgs.to(device), masks.to(device)

                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)

                for i in range(min(5, preds.shape[0])):

                    original_name = str(filenames[i]).split('.')[0]
                    filename = f"{original_name}_{file_suffix}.png"

                    pred_path = os.path.join(save_dir, "pred_" + filename)
                    gt_path   = os.path.join(save_dir, "gt_" + filename)

                    save_mask(preds[i], pred_path)
                    save_mask(masks[i], gt_path)

    return best_dice

In [ ]:
# Step 1: preprocess once
processed_folds = preprocess_all_folds(dataset)

# Step 2: run full CV automatically
run_cross_validation(processed_folds)

In [ ]:
"""
#training one one fold as validation 
val_images_np, val_masks_np = processed_folds[0]

val_images = torch.tensor(val_images_np, dtype=torch.float32)
val_masks  = torch.tensor(val_masks_np, dtype=torch.float32)

train_images_np = np.concatenate([
    processed_folds[1][0],
    processed_folds[2][0]
], axis=0)

train_masks_np = np.concatenate([
    processed_folds[1][1],
    processed_folds[2][1]
], axis=0)

train_images = torch.tensor(train_images_np, dtype=torch.float32)
train_masks  = torch.tensor(train_masks_np, dtype=torch.float32)
"""

In [ ]:
"""
model = unet(1)

best_dice = train_model(
    train_images, train_masks,
    val_images, val_masks,
    model,
    fold_idx=0,
    epochs=5,
    batch_size=4
)
"""

In [ ]:
import matplotlib.pyplot as plt
import torch

def visualize_predictions(model, images, masks, device, num_samples=3):

    model.eval()

    indices = torch.randperm(len(images))[:num_samples]

    with torch.no_grad():
        for idx in indices:

            img = images[idx].unsqueeze(0).to(device)   # add batch dim
            gt_mask = masks[idx].cpu().numpy()

            output = model(img)
            pred_mask = torch.sigmoid(output).cpu().numpy()[0, 0]

            # threshold
            pred_mask = (pred_mask > 0.5).astype(int)

            img_np = images[idx].cpu().numpy().transpose(1, 2, 0)

            # ---- PLOT ----
            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.title("Image")
            plt.imshow(img_np)
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.title("Ground Truth")
            plt.imshow(gt_mask.squeeze(), cmap='gray')
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.title("Prediction")
            plt.imshow(pred_mask, cmap='gray')
            plt.axis("off")

            plt.show()

In [ ]:
"""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load("/kaggle/working/best_model_fold_0.pth"))
model.to(device)

visualize_predictions(model, val_images, val_masks, device, num_samples=3)
"""

In [ ]:
"""
from IPython.display import FileLink

# Path to your saved model
model_path = "/kaggle/working/best_model_fold_0.pth"

# Creates a clickable download link
FileLink(model_path)
"""